# v1101 字段结构与当天多频 GRU 平台提交

上传本 Notebook 和同目录的 `v1101-model.json`，共两个 UTF-8 文本文件。
运行 `v1101-structured-fields-s42`，按验证集选定第 5 轮权重。本地判定：**未达到本地改善条件**。导出理由：用户于2026-10-22明确要求为v1101生成平台提交材料，用于评测字段结构方案；保留验证IC未晋级的原判定和第5轮EMA权重。。
JSON 包含完整数值权重、三频统计量、配置、依赖及训练／推理源码。FP32 权重以九位有效数字保存，加载后按原 dtype 恢复，并已校验逐张量字节完全一致；统计量保留原精度。
入口为 `main(datasources, start_date, end_date)`。`datasources` 提供 `bar1m`、`bar5m`、`bar30m` 的实际行情表名。导入不查询行情；推理仅使用授权股票池和三频行情，不读取标签或训练。日期字符串的结束日包含全天，时间戳的结束分钟也包含在内。
训练目标为完整截面有限标签的 z-score（总体标准差，`ddof=0`），使用 Huber `delta=2` 和 batch 512；另设权重 0.2、delta=1 的独立排名 zscore Huber 辅助头；推理只取收益头。标准化仅用于监督目标。按验证 IC 选模；动态早停耐心为最佳 epoch 的 60% 向上取整、最少 4 轮、最多 12 轮，严格创新高即重置。
本运行从头使用全局随机股票日和独立的三频当天 FP16 输入缓存训练，没有迁移或重训其他模型权重。推理保留冻结配置指定的 FP16 舍入及 FP32 极值恢复，模型计算为 FP32；训练时 GRU 状态递推也保持 FP32。
训练时原始参数由 AdamW 更新，每步随后更新 FP32 EMA 副本；首轮逐步复制原参数，之后 decay=0.9999。每轮按 EMA 的验证 IC 选择权重，本包固定加载第 5 轮、累计 75615 次更新的 EMA 参数，推理期间不更新参数平均。复现源码包含 EMA 状态、优化器及随机状态的恢复逻辑。
`factor_scale=1`，直接输出原始模型分数，单位为标准化目标单位，未校准为预测收益。公共评估器对原始收益标签计算的 MSE／MAE 不代表标准化目标上的拟合误差；IC／Rank IC 仍使用官方原始标签。
三路窗口为 480／288／168 根，分别保留此前 1／5／20 个交易日及当天已结束的前缀。每频保留原 75→192 线性投影，并增加五档盘口与非盘口字段的可学习残差编码；档位／通道交互只发生于同一 Bar 内。随后使用日内和跨日 GRU、两层同一时点跨频交互，融合宽度 384，输出收益／排名双头；平台仅取收益头。无 LayerNorm，总参数为 2,296,710。
推理依赖：numpy>=1.26,<3, pandas>=2.2,<3, pyarrow>=14, torch>=2.6,<3，以及平台 `dai`。有 CUDA 时使用 GPU，否则使用 CPU；平台环境与成绩须实际验证。
训练复现：先安装 JSON 所附 `requirements-training.txt` 对应依赖，提供授权原始 Parquet，再显式调用 `reproduce("reproduced-v1101", "/path/to/data", training_confirmed=True)`。
复现工作目录必须不存在；该调用会生成预处理数据并从头全局训练当前配置。硬件和软件环境差异可能影响重新训练的数值结果；冻结权重可直接用于推理。仅执行下方调试单元格不会训练。
本地评估摘要与原权重 SHA-256 见 JSON 的 `metadata`；逐文件哈希及本地入口验证记录位于仓库 `experiments/v1101-structured-fields-s42-fast-submission.json`，无需上传。

本包采用股票日共享编码：每只股票一次取窗并复用八时点历史表示，仍使用冻结的 FP32 GRU 权重及原归一化。窗口网格仅保存请求目标股票，截面归一化仍处理完整行情与原股票池；运行时只返回请求键。训练源码及原模型结构不变，推理批量形状变化的数值影响须以独立验证记录为准。


In [ ]:
"""两文件提交的 Notebook 入口模板；导入时不读取数据、不创建附件。"""

_RUNTIME = None


def _bundle_directory():
    from pathlib import Path
    return Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()


def _read_bundle():
    import json
    raw = (_bundle_directory() / "v1101-model.json").read_bytes()
    return json.loads(raw.decode("utf-8-sig"))


def _load_modules(bundle, paths):
    import torch
    import builtins
    from pathlib import Path
    from types import ModuleType
    modules = {}

    def local_import(name, globals=None, locals=None, fromlist=(), level=0):
        if level == 0 and name in modules:
            return modules[name]
        return builtins.__import__(name, globals, locals, fromlist, level)

    for relative in paths:
        entry = bundle["sources"][relative]
        name = Path(relative).stem
        module = ModuleType(name)
        module.__file__ = str(_bundle_directory() / Path(relative).name)
        module.__dict__["__builtins__"] = dict(vars(builtins), __import__=local_import)
        modules[name] = module
        exec(compile(entry["text"], module.__file__, "exec"), module.__dict__)
    return modules


def _runtime():
    global _RUNTIME
    if _RUNTIME is None:
        bundle = _read_bundle()
        modules = _load_modules(bundle, bundle["runtime_sources"])
        _RUNTIME = bundle, modules
    return _RUNTIME


def main(datasources, start_date, end_date):
    bundle, modules = _runtime()
    return modules["v1101_predict"].predict(
        datasources, start_date, end_date, model_dir=_bundle_directory(), bundle=bundle,
    )


def reproduce(workspace, data_root, *, training_confirmed=False):
    """显式调用才启动预处理和训练；workspace 必须是不存在的新目录。"""
    if not training_confirmed:
        raise ValueError("v1101 训练复现尚待用户明确确认")
    bundle = _read_bundle()
    modules = _load_modules(bundle, ["solutions/v1101/v1101_reproduce.py"])
    return modules["v1101_reproduce"].reproduce(bundle, workspace, data_root,
                                           training_confirmed=True)


In [ ]:
if __name__ == "__main__":
    result = main(
        {f: "cpt_jyc_2026_stock_" + f for f in ["bar1m", "bar5m", "bar30m"]},
        "2024-12-30", "2024-12-31",
    )
    import numpy as np
    import pandas as pd
    assert list(result.columns) == ["date", "instrument", "factor"]
    assert not result.empty and result["factor"].dtype == np.dtype("float64")
    assert pd.api.types.is_datetime64_any_dtype(result["date"])
    assert result["instrument"].map(lambda value: isinstance(value, str)).all()
    assert not result[["date", "instrument"]].isna().any().any()
    assert not result.duplicated(["date", "instrument"]).any()
    assert np.isfinite(result["factor"]).all()
    print("输出行数：", len(result))
    print(result.dtypes)
    display(result.head())
